# M08 · Tool use e agentes: prática

1. Function calling passo a passo.
2. Um agente do zero em 25 linhas.
3. Workflows: routing e paralelização.
4. Um agente que escreve e executa código pandas.
5. O mesmo agente em LangGraph e em PydanticAI.

**Requisitos:** `uv sync --extra agentes`, `ANTHROPIC_API_KEY`.

Lição de apoio: [licao.html](licao.html)

In [ ]:
import json, os, sys, asyncio
import pandas as pd
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()
MODELO = os.getenv("MODELO", "claude-haiku-5-5")
MODELO_AGENTE = os.getenv("MODELO_AGENTE", "claude-sonnet-5-5")

## 1. Function calling

Uma «base de dados» de encomendas e duas ferramentas.

In [ ]:
ENCOMENDAS = {
    "LL-20931": {"estado": "em trânsito", "transportadora": "CTT Expresso", "previsao": "2026-11-12", "cliente": "Marta Silva"},
    "LL-20988": {"estado": "entregue", "transportadora": "DPD", "previsao": "2026-11-05", "cliente": "Rui Costa"},
}

def consultar_encomenda(numero: str) -> str:
    e = ENCOMENDAS.get(numero.upper().strip())
    if not e:
        return f"Encomenda {numero} não existe. Confirma o número: formato LL- seguido de 5 dígitos."
    return json.dumps({k: v for k, v in e.items() if k != "cliente"}, ensure_ascii=False)

def calcular_portes(destino: str, valor_compra: float) -> str:
    tabela = {"continente": (3.90, 50), "açores": (6.90, 80), "madeira": (6.90, 80), "espanha": (7.90, None)}
    d = destino.lower()
    if d not in tabela:
        return f"Destino '{destino}' não suportado. Opções: {', '.join(tabela)}."
    preco, gratis = tabela[d]
    return f"{0 if gratis and valor_compra >= gratis else preco:.2f} €"

FERRAMENTAS = [
    {"name": "consultar_encomenda", "description": "Estado, transportadora e data prevista de uma encomenda. Usa quando o cliente pergunta por uma encomenda.",
     "input_schema": {"type": "object", "properties": {"numero": {"type": "string", "description": "Formato LL-NNNNN"}}, "required": ["numero"]}},
    {"name": "calcular_portes", "description": "Custo de envio para um destino e valor de compra.",
     "input_schema": {"type": "object", "properties": {
         "destino": {"type": "string", "enum": ["continente", "açores", "madeira", "espanha"]},
         "valor_compra": {"type": "number", "description": "Valor da compra em euros"}}, "required": ["destino", "valor_compra"]}},
]
FUNCOES = {"consultar_encomenda": consultar_encomenda, "calcular_portes": calcular_portes}

r = client.messages.create(model=MODELO, max_tokens=500, tools=FERRAMENTAS,
                           messages=[{"role": "user", "content": "Onde está a minha encomenda LL-20931?"}])
print(r.stop_reason)
for b in r.content:
    print(b)

O modelo pediu a ferramenta. Executamos e devolvemos o resultado com o mesmo `tool_use_id`:

In [ ]:
pedido = next(b for b in r.content if b.type == "tool_use")
resultado = FUNCOES[pedido.name](**pedido.input)
r2 = client.messages.create(model=MODELO, max_tokens=500, tools=FERRAMENTAS, messages=[
    {"role": "user", "content": "Onde está a minha encomenda LL-20931?"},
    {"role": "assistant", "content": r.content},
    {"role": "user", "content": [{"type": "tool_result", "tool_use_id": pedido.id, "content": resultado}]},
])
print(r2.content[0].text)

## 2. Um agente do zero

In [ ]:
def agente(pergunta, ferramentas=FERRAMENTAS, funcoes=FUNCOES, system="És o assistente da Loja Lusa.", modelo=MODELO, max_passos=8, verbose=True):
    msgs = [{"role": "user", "content": pergunta}]
    for passo in range(max_passos):
        r = client.messages.create(model=modelo, max_tokens=1500, system=system, tools=ferramentas, messages=msgs)
        msgs.append({"role": "assistant", "content": r.content})
        if r.stop_reason != "tool_use":
            return "".join(b.text for b in r.content if b.type == "text")
        res = []
        for b in r.content:
            if b.type == "tool_use":
                try:
                    out = funcoes[b.name](**b.input)
                except Exception as e:                      # o erro volta ao modelo, que pode corrigir
                    out = f"ERRO: {e}"
                if verbose:
                    print(f"  🔧 {b.name}({b.input}) → {out}")
                res.append({"type": "tool_result", "tool_use_id": b.id, "content": str(out)})
        msgs.append({"role": "user", "content": res})
    return "Limite de passos atingido."

print(agente("Tenho duas encomendas, LL-20931 e LL-2098. Como estão? E se comprar mais 60 € para os Açores, quanto pago de portes?"))

Repara: o modelo chamou várias ferramentas em paralelo no mesmo passo e recuperou do erro do número mal escrito.

**Exercício 2.1.** Acrescenta uma ferramenta `abrir_reclamacao(numero, motivo)` que só deve ser usada quando o cliente pede explicitamente. Testa com uma mensagem ambígua («a encomenda está atrasada...»): o agente abre a reclamação sem perguntar? Ajusta a descrição até ele pedir confirmação.

## 3. Workflows

### 3.1 Routing

In [ ]:
from pydantic import BaseModel
from typing import Literal
import instructor

ic = instructor.from_anthropic(client)

class Rota(BaseModel):
    destino: Literal["encomendas", "devolucoes", "humano"]

PROMPTS_ROTA = {
    "encomendas": "És especialista em entregas da Loja Lusa. Usa as ferramentas para consultar encomendas.",
    "devolucoes": "És especialista em devoluções da Loja Lusa: 30 dias, artigo sem uso, devolução gratuita nos CTT.",
}

def atender(mensagem):
    rota = ic.messages.create(model=MODELO, max_tokens=50, response_model=Rota, messages=[{"role": "user", "content":
        f"Encaminha a mensagem: encomendas (estado, entregas, portes), devolucoes (trocas, reembolsos), humano (reclamações graves, outros).\n<msg>{mensagem}</msg>"}]).destino
    if rota == "humano":
        return rota, "Vou passar a sua mensagem a um colega, que responde em até 24h úteis."
    return rota, agente(mensagem, system=PROMPTS_ROTA[rota], verbose=False)

for m in ["Onde está a LL-20988?", "Quero devolver um edredão que não gostei", "Isto é uma vergonha, vou fazer queixa à DECO!"]:
    print(atender(m), "\n")

### 3.2 Paralelização

Analisar várias avaliações de clientes ao mesmo tempo com o cliente assíncrono.

In [ ]:
from anthropic import AsyncAnthropic
aclient = AsyncAnthropic()

avaliacoes = [
    "Lençóis ótimos, mas a entrega demorou 8 dias.",
    "A frigideira descolou ao fim de duas semanas. Péssimo.",
    "Atendimento impecável, resolveram a troca num dia.",
    "Preço bom, qualidade média. O site é confuso no telemóvel.",
]

class Analise(BaseModel):
    sentimento: Literal["positivo", "neutro", "negativo"]
    temas: list[Literal["produto", "entrega", "atendimento", "preço", "site"]]

aic = instructor.from_anthropic(aclient)

async def analisar(t):
    return await aic.messages.create(model=MODELO, max_tokens=200, response_model=Analise,
                                     messages=[{"role": "user", "content": f"Analisa a avaliação: {t}"}])

res = await asyncio.gather(*[analisar(t) for t in avaliacoes])
pd.DataFrame([{"avaliação": t, **r.model_dump()} for t, r in zip(avaliacoes, res)])

**Exercício 3.1.** Implementa o padrão avaliador-otimizador: um LLM escreve a resposta a uma reclamação, outro avalia segundo 4 critérios (empatia, solução concreta, sem promessas que a empresa não cumpre, menos de 120 palavras) e devolve feedback; repete até passar ou 3 iterações.

## 4. Agente que executa código

Usamos a `Sandbox` de `use-case/sandbox.py` (subprocesso com limites; ver avisos na lição) e os dados sintéticos de vendas.

In [ ]:
sys.path.insert(0, "use-case")
from gerar_vendas import gerar
from agente_dados import analisar

os.makedirs("use-case/dados", exist_ok=True)
gerar().to_csv("use-case/dados/vendas.csv", index=False)

out = analisar("use-case/dados/vendas.csv", "Que categoria cresceu mais em 2025 face a 2024, e em que loja?", pasta_trabalho="use-case/trabalho")
print("\n", out["resposta"], "\n", out["passos"], "passos", out["uso"])

In [ ]:
from IPython.display import Image
Image(out["grafico"]) if out["grafico"] else None

Os dados têm padrões conhecidos (ver `gerar_vendas.py`): Cozinha cresce cerca de 25% e a loja online mais do que as físicas. O agente encontrou-os? Tratou os valores em falta e as devoluções (valores negativos)?

**Exercício 4.1.** Faz 5 perguntas diferentes e regista passos, tokens e se a resposta está certa. Que perguntas o fazem falhar?

## 5. Frameworks

### 5.1 LangGraph

O `create_react_agent` monta o ciclo por nós. Por baixo, é um grafo com um nó «modelo» e um nó «ferramentas» e uma aresta condicional.

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

@tool
def consultar_encomenda_lc(numero: str) -> str:
    """Estado, transportadora e data prevista de uma encomenda (formato LL-NNNNN)."""
    return consultar_encomenda(numero)

@tool
def calcular_portes_lc(destino: str, valor_compra: float) -> str:
    """Custo de envio. destino: continente, açores, madeira ou espanha."""
    return calcular_portes(destino, valor_compra)

grafo = create_react_agent(ChatAnthropic(model=MODELO), [consultar_encomenda_lc, calcular_portes_lc],
                           prompt="És o assistente da Loja Lusa.")
estado = grafo.invoke({"messages": [("user", "Como está a LL-20931 e quanto custam os portes de 40 € para a Madeira?")]})
for m in estado["messages"]:
    print(type(m).__name__, ":", (m.content if isinstance(m.content, str) else m.content)[:200] if m.content else m.tool_calls)

### 5.2 PydanticAI

Agentes tipados: o resultado é um modelo Pydantic e as dependências (aqui, a «base de dados») são injetadas.

In [ ]:
from dataclasses import dataclass
from pydantic_ai import Agent, RunContext

@dataclass
class Deps:
    encomendas: dict

class RespostaCliente(BaseModel):
    resposta: str
    precisa_humano: bool

agente_pai = Agent(f"anthropic:{MODELO}", deps_type=Deps, output_type=RespostaCliente,
                   system_prompt="És o assistente da Loja Lusa. Marca precisa_humano=True para reclamações.")

@agente_pai.tool
def estado_encomenda(ctx: RunContext[Deps], numero: str) -> str:
    """Estado de uma encomenda (formato LL-NNNNN)."""
    e = ctx.deps.encomendas.get(numero)
    return json.dumps(e, ensure_ascii=False) if e else "não existe"

r = await agente_pai.run("A LL-20988 diz entregue mas eu não recebi nada!", deps=Deps(ENCOMENDAS))
r.output

## Use case: agente analista de dados

```bash
cd modulos/08-tool-use-e-agentes/use-case
uv run python gerar_vendas.py
uv run python agente_dados.py dados/vendas.csv "Qual o peso das devoluções por categoria e como evoluiu?"
```

**Desafio:** troca a `Sandbox` por um contentor Docker sem rede (`docker run --rm --network none -v $PWD/trabalho:/w python:3.12-slim ...`) e cria uma interface Streamlit que mostra o código de cada passo e o gráfico final.